### LLM Inference Benchmarking Assignment (35 points)

#### Overview
In this assignment, you will learn how to benchmark Large Language Model (LLM) inference using vLLM, a high-performance inference engine. You will use a small model (OPT-125M) to understand the basics of throughput measurement and the impact of different parameters on inference speed.

#### Environment Setup (4 points)

- Successfully install all required packages


First, install the required packages:

In [1]:
import sys, torch, vllm
print(sys.version)
print(vllm.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))
from vllm import LLM, SamplingParams

3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
0.19.0
True
1
NVIDIA A100-SXM4-40GB


In [2]:
# install via pip
%pip install -q vllm transformers pandas matplotlib seaborn


Note: you may need to restart the kernel to use updated packages.


Import the necessary libraries:

In [3]:
# Import necessary libraries
import dataclasses
import random
import time
from typing import Dict, List

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

MODEL_NAME = "facebook/opt-125m"
RANDOM_SEED = 301
random.seed(RANDOM_SEED)

_TOKENIZER_CACHE: Dict[str, AutoTokenizer] = {}
_LLM_CACHE: Dict[tuple, LLM] = {}


def get_tokenizer(model_name: str = MODEL_NAME):
    if model_name not in _TOKENIZER_CACHE:
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        if tokenizer.pad_token is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        _TOKENIZER_CACHE[model_name] = tokenizer
    return _TOKENIZER_CACHE[model_name]


def get_llm(
    model_name: str,
    tensor_parallel_size: int,
    gpu_memory_utilization: float,
    max_num_batched_tokens: int,
):
    cache_key = (
        model_name,
        tensor_parallel_size,
        gpu_memory_utilization,
        max_num_batched_tokens,
    )
    if cache_key not in _LLM_CACHE:
        _LLM_CACHE[cache_key] = LLM(
            model=model_name,
            tensor_parallel_size=tensor_parallel_size,
            gpu_memory_utilization=gpu_memory_utilization,
            max_num_batched_tokens=max_num_batched_tokens,
        )
    return _LLM_CACHE[cache_key]


if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook requires a CUDA-enabled Linux environment. "
        "Run it on the HPC GPU node before executing the benchmark cells."
    )

sns.set_theme(style="whitegrid")

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))
print("Default model:", MODEL_NAME)
print("All libraries imported successfully!")


CUDA available: True
CUDA device count: 1
GPU name: NVIDIA A100-SXM4-40GB
Default model: facebook/opt-125m
All libraries imported successfully!


#### Part 1: Creating Sample Requests (7 points)

- Implement the create_synthetic_requests function correctly

We'll create a function to generate sample requests for benchmarking:

In [4]:
@dataclasses.dataclass
class SampleRequest:
    """A class representing a single inference request for benchmarking."""
    prompt: str
    prompt_len: int
    expected_output_len: int


def create_synthetic_requests(
    tokenizer,
    num_requests: int,
    input_len: int,
    output_len: int
) -> List[SampleRequest]:
    """Create synthetic requests for benchmarking.

    Args:
        tokenizer: The tokenizer to use
        num_requests: Number of requests to generate
        input_len: Desired input length in tokens
        output_len: Desired output length in tokens

    Returns:
        List of SampleRequest objects
    """
    if num_requests <= 0:
        raise ValueError("num_requests must be positive.")
    if input_len <= 0 or output_len <= 0:
        raise ValueError("input_len and output_len must be positive.")

    seed_text = (
        "Benchmark the throughput of this language model request. "
        "Measure prompt processing, output generation, and batching behavior. "
        "Return a concise continuation about inference efficiency and scaling. "
    )
    filler_ids = tokenizer.encode(seed_text, add_special_tokens=False)
    if not filler_ids:
        raise ValueError("The tokenizer produced an empty token sequence for the seed text.")

    requests: List[SampleRequest] = []
    for idx in range(num_requests):
        request_text = f" Request {idx}: analyze latency, throughput, memory, and scheduling."
        request_ids = tokenizer.encode(request_text, add_special_tokens=False)
        token_ids = list(request_ids)

        while len(token_ids) < input_len:
            token_ids.extend(filler_ids)
        token_ids = token_ids[:input_len]

        tokens = tokenizer.convert_ids_to_tokens(token_ids)
        prompt = tokenizer.convert_tokens_to_string(tokens)
        verified_ids = tokenizer.encode(prompt, add_special_tokens=False)

        if verified_ids != token_ids:
            raise ValueError(
                f"Prompt tokenization drifted for request {idx}: "
                f"expected {len(token_ids)} tokens, got {len(verified_ids)}."
            )

        requests.append(
            SampleRequest(
                prompt=prompt,
                prompt_len=len(verified_ids),
                expected_output_len=output_len,
            )
        )

    return requests


#### Part 2: Implementing the Benchmark (10 points)
- Implement the run_benchmark function correctly

Create the main benchmarking function:

In [5]:
def _execute_benchmark(
    requests: List[SampleRequest],
    model_name: str,
    tensor_parallel_size: int = 1,
    gpu_memory_utilization: float = 0.9,
    max_num_batched_tokens: int = 2048,
    n: int = 1
) -> Dict[str, float]:
    if not requests:
        raise ValueError("requests must contain at least one SampleRequest.")
    if n <= 0:
        raise ValueError("n must be positive.")

    llm = get_llm(
        model_name=model_name,
        tensor_parallel_size=tensor_parallel_size,
        gpu_memory_utilization=gpu_memory_utilization,
        max_num_batched_tokens=max_num_batched_tokens,
    )
    tokenizer = get_tokenizer(model_name)

    max_model_len = getattr(llm.llm_engine.model_config, "max_model_len", None)
    prompts = []
    sampling_params = []

    for request in requests:
        if request.prompt_len <= 0 or request.expected_output_len <= 0:
            raise ValueError("Request token lengths must be positive.")
        if max_model_len is not None and request.prompt_len + request.expected_output_len > max_model_len:
            raise ValueError(
                f"Request exceeds the model context window: "
                f"{request.prompt_len} prompt tokens + {request.expected_output_len} output tokens "
                f"> {max_model_len}."
            )

        prompts.append(request.prompt)
        sampling_params.append(
            SamplingParams(
                n=n,
                temperature=0.0,
                top_p=1.0,
                ignore_eos=True,
                max_tokens=request.expected_output_len,
            )
        )

    start = time.perf_counter()
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    elapsed = time.perf_counter() - start

    if elapsed <= 0:
        raise RuntimeError("Measured benchmark time must be positive.")

    total_input_tokens = sum(request.prompt_len for request in requests)
    total_output_tokens = 0
    for output in outputs:
        for completion in output.outputs:
            if hasattr(completion, "token_ids") and completion.token_ids is not None:
                total_output_tokens += len(completion.token_ids)
            else:
                total_output_tokens += len(
                    tokenizer(completion.text, add_special_tokens=False).input_ids
                )

    return {
        "elapsed_time": elapsed,
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "total_tokens": total_input_tokens + total_output_tokens,
        "num_requests": len(requests),
    }


def run_benchmark(
    requests: List[SampleRequest],
    model_name: str,
    tensor_parallel_size: int = 1,
    gpu_memory_utilization: float = 0.9,
    max_num_batched_tokens: int = 2048,
    n: int = 1
) -> float:
    """Run inference benchmark using vLLM.

    Args:
        requests: List of requests to process
        model_name: Name of the model to benchmark
        tensor_parallel_size: Number of GPUs for tensor parallelism
        gpu_memory_utilization: Target GPU memory utilization
        max_num_batched_tokens: Maximum number of tokens in a batch
        n: Number of sequences to generate per prompt

    Returns:
        Elapsed time in seconds
    """
    result = _execute_benchmark(
        requests=requests,
        model_name=model_name,
        tensor_parallel_size=tensor_parallel_size,
        gpu_memory_utilization=gpu_memory_utilization,
        max_num_batched_tokens=max_num_batched_tokens,
        n=n,
    )
    return result["elapsed_time"]


#### Part 3: Running the Benchmark (14 points)

- Experiments and Analysis (8 points)

  - Run the benchmark with different batch sizes (3 points)

In [6]:
def experiment_batch_sizes(model_name: str, batch_sizes: List[int]) -> pd.DataFrame:
    """Run benchmark with different batch sizes.

    Args:
        model_name: Name of the model to benchmark
        batch_sizes: List of batch sizes to test

    Returns:
        DataFrame with results
    """
    tokenizer = get_tokenizer(model_name)
    rows = []

    for batch_size in batch_sizes:
        requests = create_synthetic_requests(
            tokenizer=tokenizer,
            num_requests=batch_size,
            input_len=128,
            output_len=64,
        )
        result = _execute_benchmark(requests=requests, model_name=model_name)
        rows.append(
            {
                "batch_size": batch_size,
                "elapsed_time": result["elapsed_time"],
                "throughput": result["total_tokens"] / result["elapsed_time"],
                "output_throughput": result["total_output_tokens"] / result["elapsed_time"],
                "request_throughput": result["num_requests"] / result["elapsed_time"],
            }
        )

    return pd.DataFrame(rows)


  - Run the benchmark with different input/output lengths (3 points)


In [7]:
def experiment_sequence_lengths(model_name: str, lengths: List[int]) -> pd.DataFrame:
    """Run benchmark with different input/output lengths.

    Args:
        model_name: Name of the model to benchmark
        lengths: List of sequence lengths to test

    Returns:
        DataFrame with results
    """
    tokenizer = get_tokenizer(model_name)
    rows = []

    for length in lengths:
        requests = create_synthetic_requests(
            tokenizer=tokenizer,
            num_requests=16,
            input_len=length,
            output_len=length,
        )
        result = _execute_benchmark(requests=requests, model_name=model_name)
        rows.append(
            {
                "sequence_length": length,
                "elapsed_time": result["elapsed_time"],
                "throughput": result["total_tokens"] / result["elapsed_time"],
                "output_throughput": result["total_output_tokens"] / result["elapsed_time"],
                "request_throughput": result["num_requests"] / result["elapsed_time"],
            }
        )

    return pd.DataFrame(rows)


  - Run the benchmark with different numbers of requests (2 points)


In [8]:
def experiment_num_requests(model_name: str, request_counts: List[int]) -> pd.DataFrame:
    """Run benchmark with different numbers of requests.

    Args:
        model_name: Name of the model to benchmark
        request_counts: List of request counts to test

    Returns:
        DataFrame with results
    """
    tokenizer = get_tokenizer(model_name)
    rows = []

    for request_count in request_counts:
        requests = create_synthetic_requests(
            tokenizer=tokenizer,
            num_requests=request_count,
            input_len=128,
            output_len=64,
        )
        result = _execute_benchmark(requests=requests, model_name=model_name)
        rows.append(
            {
                "num_requests": request_count,
                "elapsed_time": result["elapsed_time"],
                "throughput": result["total_tokens"] / result["elapsed_time"],
                "output_throughput": result["total_output_tokens"] / result["elapsed_time"],
                "request_throughput": result["num_requests"] / result["elapsed_time"],
            }
        )

    return pd.DataFrame(rows)


- Analysis and Visualization (6 points)
    - Create a graph showing the relationship between batch size and throughput (4 points)

Now let's put everything together and run the benchmark:


In [9]:
def run_all_experiments(model_name: str = MODEL_NAME):
    """Run all experiments and create visualizations."""

    # Experiment configurations
    batch_sizes = [1, 2, 4, 8, 16, 32]
    sequence_lengths = [32, 64, 128, 256, 512]
    request_counts = [10, 50, 100, 200, 500]

    # Run experiments
    print("Running batch size experiments...")
    batch_results = experiment_batch_sizes(model_name, batch_sizes)

    print("Running sequence length experiments...")
    length_results = experiment_sequence_lengths(model_name, sequence_lengths)

    print("Running request count experiments...")
    request_results = experiment_num_requests(model_name, request_counts)

    # Create visualizations
    plt.figure(figsize=(15, 5))

    # Batch size vs throughput
    plt.subplot(1, 3, 1)
    sns.lineplot(data=batch_results, x="batch_size", y="throughput", marker="o")
    plt.title("Batch Size vs Throughput")
    plt.xlabel("Batch Size")
    plt.ylabel("Tokens/second")
    plt.grid(True, alpha=0.3)

    # Sequence length vs throughput
    plt.subplot(1, 3, 2)
    sns.lineplot(data=length_results, x="sequence_length", y="throughput", marker="o")
    plt.title("Sequence Length vs Throughput")
    plt.xlabel("Sequence Length")
    plt.ylabel("Tokens/second")
    plt.grid(True, alpha=0.3)

    # Number of requests vs throughput
    plt.subplot(1, 3, 3)
    sns.lineplot(data=request_results, x="num_requests", y="throughput", marker="o")
    plt.title("Number of Requests vs Throughput")
    plt.xlabel("Number of Requests")
    plt.ylabel("Tokens/second")
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return batch_results, length_results, request_results


# Run experiments and store results
batch_results, length_results, request_results = run_all_experiments()


Running batch size experiments...


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

ValueError: Prompt tokenization drifted for request 0: expected 128 tokens, got 125.

 - Write a clear analysis of the results (2 points)


After running the notebook on the HPC GPU node, interpret the results in terms of throughput and batching:

- Throughput should generally increase as batch size grows, then flatten when the GPU is saturated.
- Longer input/output sequences should reduce throughput because the model spends more time on prefill and autoregressive decoding.
- Increasing the number of requests should improve utilization up to a point, after which scheduler and memory pressure limit gains.
- Compare the best-performing settings across the three experiments and connect them back to vLLM's batching behavior.
- Note any resource constraints you observed on the cluster, such as memory limits, queue time, or practical upper bounds on request volume.


In [ ]:
summary_rows = [
    {
        "experiment": "batch_size",
        "best_setting": int(batch_results.loc[batch_results["throughput"].idxmax(), "batch_size"]),
        "best_throughput": float(batch_results["throughput"].max()),
    },
    {
        "experiment": "sequence_length",
        "best_setting": int(length_results.loc[length_results["throughput"].idxmax(), "sequence_length"]),
        "best_throughput": float(length_results["throughput"].max()),
    },
    {
        "experiment": "num_requests",
        "best_setting": int(request_results.loc[request_results["throughput"].idxmax(), "num_requests"]),
        "best_throughput": float(request_results["throughput"].max()),
    },
]

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

best_batch = batch_results.loc[batch_results["throughput"].idxmax()]
best_length = length_results.loc[length_results["throughput"].idxmax()]
best_request_count = request_results.loc[request_results["throughput"].idxmax()]

print(
    f"Best batch-size throughput: {best_batch['throughput']:.2f} tokens/s "
    f"at batch size {int(best_batch['batch_size'])}."
)
print(
    f"Best sequence-length throughput: {best_length['throughput']:.2f} tokens/s "
    f"at sequence length {int(best_length['sequence_length'])}."
)
print(
    f"Best request-count throughput: {best_request_count['throughput']:.2f} tokens/s "
    f"at {int(best_request_count['num_requests'])} concurrent requests."
)


## 